# Lab 11: Zero-Bus Streaming with Zerobus

Implementing an event-driven ingestion pipeline **without a centralized message broker** using Databricks Zerobus (push-based single-sink model). The pipeline streams user movie consumption events — `movie_started`, `watch_progress`, `rating_submitted`, `watchlist_added` — into `main.lab_data.movie_stream_events` (Delta table with Change Data Feed enabled) and guarantees **idempotent ingestion** to handle network retries and at-least-once delivery.

**Notebook:** `01_zerobus_producer_and_idempotency.ipynb`

---
## Prerequisites

Before running this lab, you need:
1. **Databricks Zerobus Ingest SDK** (installed below)
2. **Service Principal** with credentials (CLIENT_ID and CLIENT_SECRET)
3. **Workspace URL** and **Zerobus endpoint** for your region
4. **Unity Catalog permissions** granted to the service principal

In [0]:
%pip install databricks-zerobus-ingest-sdk

In [0]:
# ─────────────────────────────────────────────────────────────────────
#  Zerobus Ingest Configuration
# ─────────────────────────────────────────────────────────────────────

CATALOG = "main"
SCHEMA = "lab_data"

# workspace URL
DATABRICKS_WORKSPACE_URL = "https://dbc-825edb48-78cb.cloud.databricks.com"

# Zerobus server endpoint
ZEROBUS_SERVER_ENDPOINT = "https://7474649823137075.zerobus.us-east-2.cloud.databricks.com"

# Service principal credentials — loaded from Databricks Secrets 
CLIENT_ID = dbutils.secrets.get(scope="zerobus-lab11", key="client-id")
CLIENT_SECRET = dbutils.secrets.get(scope="zerobus-lab11", key="client-secret")

# Target table
TARGET_TABLE = f"{CATALOG}.{SCHEMA}.movie_stream_events"

print("Zerobus configuration loaded. Edit the values above before running ingestion.")

In [0]:
# Grant permissions to the service principal for Zerobus Ingest
# CLIENT_ID is loaded from Databricks Secrets (scope: zerobus-lab11)
for stmt in [
    f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{CLIENT_ID}`",
    f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.{SCHEMA} TO `{CLIENT_ID}`",
    f"GRANT MODIFY, SELECT ON TABLE {CATALOG}.{SCHEMA}.movie_stream_events TO `{CLIENT_ID}`",
]:
    spark.sql(stmt)
    print(f"✅ Executed: {stmt}")

---
## Part 1: Setup — Delta Table with Change Data Feed

Create the target Delta table `movie_stream_events` with **Change Data Feed (CDF)** enabled. CDF tracks row-level changes (inserts, updates, deletes) which is essential for downstream CDC consumers and audit trails in a Zero-Bus architecture.

**Zerobus Ingest requirements:**
- The table must be a **managed Delta table** (not external).
- Use `CREATE TABLE IF NOT EXISTS` — Zerobus does not support recreating a target table once it's linked to a stream.
- The table schema defines what each pushed record must contain (Zerobus validates against it).

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.movie_stream_events (
        event_id STRING,
        user_id STRING,
        movie_title STRING,
        event_type STRING,
        watch_time_seconds INT,
        event_timestamp TIMESTAMP,
        ingested_at TIMESTAMP
    )
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")
print("Table ready for Zerobus Ingest.")

---
## Part 2: Event Generation with Intentional Duplicates

Generate synthetic movie-streaming events and deliberately duplicate the first two `event_id` values to simulate **at-least-once delivery** — the real-world scenario where a producer re-sends events it didn't receive an ACK for due to network instability.

These events will be **pushed to the Delta table via the Zerobus Ingest SDK** (gRPC streaming), not through Spark DataFrames or MERGE INTO.

In [0]:
import uuid
import random
from datetime import datetime

def generate_movie_events(num_events: int = 5, inject_duplicates: bool = True) -> list:
    movie_titles = ["Inception", "The Dark Knight", "Interstellar", "Avatar", "Titanic"]
    event_types = ["movie_started", "watch_progress", "rating_submitted", "watchlist_added"]
    
    current_time_us = int(datetime.now().timestamp() * 1000000)
    events = []
    
    for _ in range(num_events):
        events.append({
            "event_id": str(uuid.uuid4()),
            "user_id": f"user_{random.randint(100, 105)}",
            "movie_title": random.choice(movie_titles),
            "event_type": random.choice(event_types),
            "watch_time_seconds": random.randint(30, 7200),
            "event_timestamp": current_time_us
        })
    
    if inject_duplicates and len(events) >= 2:
        events.append(events[0].copy())
        events.append(events[1].copy())
        print(f"⚠️  Generated {len(events)} events (including 2 duplicates).")
    else:
        print(f"✅ Generated {len(events)} unique events.")
    
    return events

batch_events = generate_movie_events(num_events=5, inject_duplicates=True)

---
## Part 3: Push-Based Ingestion via Zerobus Ingest SDK

Replace the previous Spark `MERGE INTO` approach with **real Zerobus Ingest** — a push-based streaming API that writes records directly into the Delta table over gRPC.

**How it works:**
1. Open a stream to the Zerobus server endpoint using the SDK.
2. Push each event record via `ingest_record_offset()` — records are validated against the table schema and made durable.
3. Call `flush()` to block until all pending records are confirmed durable.
4. Close the stream.

**Important:** Zerobus provides **at-least-once delivery** and only performs INSERTs (no MERGE/upsert). Duplicates from retries are expected and are handled by a **downstream dedup view** (see next cell).

In [0]:
from zerobus.sdk.sync import ZerobusSdk
from zerobus.sdk.shared import TableProperties
from datetime import datetime

def ingest_events_zerobus(events_data: list, target_table: str = TARGET_TABLE) -> None:
    sdk = ZerobusSdk(ZEROBUS_SERVER_ENDPOINT, DATABRICKS_WORKSPACE_URL)
    table_properties = TableProperties(target_table)
    stream = sdk.create_stream(CLIENT_ID, CLIENT_SECRET, table_properties)
    
    try:
        ingested_at_us = int(datetime.now().timestamp() * 1000000)
        for event in events_data:
            record = event.copy()
            record["ingested_at"] = ingested_at_us
            stream.ingest_record_offset(record)
        
        stream.flush()
        print(f"📥 Pushed {len(events_data)} records via Zerobus. All confirmed durable.")
    finally:
        stream.close()

ingest_events_zerobus(batch_events)

---
## Part 4: Idempotency via Deduplication View

**The Challenge:** Zerobus Ingest provides **at-least-once delivery** semantics and only performs INSERT operations (no MERGE/upsert). This means duplicate records from network retries will appear in the raw table.

**The Solution:** Create a **deduplication view** that uses `ROW_NUMBER()` to keep only the first occurrence of each `event_id`, ordered by `ingested_at`. This gives us **exactly-once semantics** at the consumption layer without requiring MERGE operations during ingestion.

**Key Pattern:**
- **Raw table** (`movie_stream_events`): Contains all records including duplicates (at-least-once)
- **Dedup view** (`movie_stream_events_dedup`): Shows only unique records (exactly-once)
- **Downstream consumers** query the dedup view, not the raw table

In [0]:
spark.sql(f"""
    CREATE OR REPLACE VIEW {CATALOG}.{SCHEMA}.movie_stream_events_dedup AS
    SELECT event_id, user_id, movie_title, event_type, watch_time_seconds, event_timestamp, ingested_at
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY event_id ORDER BY ingested_at) AS rn
        FROM {CATALOG}.{SCHEMA}.movie_stream_events
    )
    WHERE rn = 1
""")

raw_count = spark.table(f"{CATALOG}.{SCHEMA}.movie_stream_events").count()
dedup_count = spark.table(f"{CATALOG}.{SCHEMA}.movie_stream_events_dedup").count()

print(f"📊 Raw table: {raw_count} rows (at-least-once)")
print(f"✅ Dedup view: {dedup_count} rows (exactly-once)")
print(f"🗑️  Duplicates removed: {raw_count - dedup_count}")

---
## Part 5: Testing Network Retry Behavior (Idempotency Test)

Re-send the **exact same batch** of events through Zerobus Ingest to simulate a network retry. Because Zerobus provides **at-least-once delivery**, the raw table will contain duplicates — but the **dedup view** should show the same count as before the retry, proving the system handles network retries gracefully at the consumption layer.

In [0]:
import time

print("\n🔁 Re-sending same batch (simulating network retry)...")
ingest_events_zerobus(batch_events)
time.sleep(3)

raw_count = spark.table(f"{CATALOG}.{SCHEMA}.movie_stream_events").count()
dedup_count = spark.table(f"{CATALOG}.{SCHEMA}.movie_stream_events_dedup").count()

print(f"\n📊 Raw table (after retry): {raw_count} rows")
print(f"✅ Dedup view (after retry): {dedup_count} rows")
print(f"🗑️  Duplicates from retry: {raw_count - dedup_count}")
print("*************************************************")
print("\n📋 Dedup view (exactly-once per event_id):")
display(spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.movie_stream_events_dedup ORDER BY ingested_at"))

---
## Part 6: Architecture Comparison: Kafka vs Zerobus

This comparison evaluates the two architectures across key dimensions:
- **Infrastructure**: Traditional Kafka requires dedicated broker clusters, while Zerobus is serverless
- **Cost model**: 24/7 running compute vs pay-per-GB ingestion
- **Operational overhead**: Self-managed vs fully managed
- **Use cases**: Multi-consumer pub/sub vs single-sink lakehouse ingestion

In [0]:


def print_comparison_table():
    """Print a visual side-by-side comparison of Kafka vs Zero-Bus."""
    rows = [
        ("Category",              "Traditional Kafka",                      "Databricks Zero-Bus"),
        ("────────────────────────", "──────────────────────────────────", "──────────────────────────────────"),
        ("Message Broker",        "Dedicated cluster (3+ brokers)",         "None (push-based, no broker)"),
        ("Compute Model",         "24/7 running Spark/Consumer",            "Serverless, pay-per-ingested-GB"),
        ("Scaling",               "Manual partition & broker scaling",      "Auto-scales with load"),
        ("Operational Overhead",  "High (manage, patch, monitor brokers)",  "Minimal (managed service)"),
        ("Fault Tolerance",       "Broker replication factor tuning",      "Built-in retry + downstream dedup view"),
        ("Delivery Semantics",    "At-least-once (requires app logic)",    "At-least-once + downstream dedup view"),
        ("Idle Cost",             "Always-on compute + broker infrastructure", "~$0 when idle (serverless)"),
        ("Time to Deploy",        "Days (provision, configure, test)",       "Minutes (CREATE TABLE + push)"),
        ("Best For",              "Multi-consumer, high-throughput pub/sub", "Single-sink, event-driven ingestion"),
    ]

    col_widths = [24, 40, 40]
    sep = "│"

    # Header row
    header = sep.join(f" {r:<{w-2}} " for r, w in zip(rows[0], col_widths))
    print(header)
    print("─" * len(header))

    # Data rows
    for row in rows[1:]:
        print(sep.join(f" {cell:<{w-2}} " for cell, w in zip(row, col_widths)))


def estimate_monthly_cost(events_per_day=5_000_000):
    """Estimate and visually compare monthly TCO for both architectures."""
    # Kafka: 3 brokers (m5.large) + 24/7 single-node Spark Streaming
    kafka_infra = (3 * 0.10 * 24 * 30) + (0.40 * 24 * 30)  # ~$500/міс
    # Zero-Bus: pay per ingested GB (serverless)
    gb_per_month = (events_per_day * 500 / 1e9) * 30  # ~75 GB
    zerobus_cost = gb_per_month * 0.05 + 10  # ~$15/міс

    kafka_pct = kafka_infra / (kafka_infra + zerobus_cost) * 100
    zerobus_pct = 100 - kafka_pct

    print("\n" + "=" * 78)
    print("  📊  ESTIMATED MONTHLY TCO COMPARISON")
    print("=" * 78)
    print(f"  • Traditional Kafka + 24/7 Compute : ~${kafka_infra:.2f}  ({kafka_pct:.0f}% of total)")
    print(f"  • Databricks Zero-Bus (Serverless)  : ~${zerobus_cost:.2f}  ({zerobus_pct:.0f}% of total)")
    print("=" * 78)

    # Simple ASCII bar chart
    max_bar = 50
    kafka_bar_len = int(kafka_pct / 100 * max_bar)
    zerobus_bar_len = max_bar - kafka_bar_len

    print(f"\n  Kafka     [{'█' * kafka_bar_len:<{max_bar}}] ${kafka_infra:.2f}")
    print(f"  Zero-Bus  [{'█' * zerobus_bar_len:<{max_bar}}] ${zerobus_cost:.2f}")

    savings = kafka_infra - zerobus_cost
    savings_pct = (savings / kafka_infra) * 100
    print(f"\n  💰  Estimated savings with Zero-Bus: ~${savings:.2f}/mo  ({savings_pct:.0f}% reduction)")
    print("=" * 78 + "\n")


print_comparison_table()
estimate_monthly_cost()

---
## Summary & Key Takeaways

This notebook showcased a complete **Zero-Bus streaming architecture** using Databricks Zerobus Ingest:

1. **Direct Push-Based Ingestion**
   - Events flow directly from producers to Delta tables via Zerobus SDK (gRPC)
   - No intermediate message broker (Kafka, Kinesis, etc.) required
   - Serverless, pay-per-GB pricing model

2. **At-Least-Once Delivery with Exactly-Once Semantics**
   - Zerobus guarantees at-least-once delivery (duplicates from retries are expected)
   - Idempotency achieved via downstream deduplication view using `ROW_NUMBER()`
   - Raw table contains all records; dedup view provides exactly-once semantics

3. **Production Pattern**
   - Raw table: `movie_stream_events` (at-least-once, includes duplicates)
   - Dedup view: `movie_stream_events_dedup` (exactly-once per event_id)
   - Consumers query the dedup view, not the raw table

4. **Cost & Operational Benefits**
   - ~97% cost reduction vs traditional Kafka architecture ($13.75 vs $504/month)
   - Zero operational overhead (no brokers to manage, patch, or scale)
   - Auto-scaling with load, ~$0 when idle

### When to Use Zerobus vs Kafka

**Use Zerobus when:**
- Single destination (lakehouse) for event data
- Event-driven ingestion into Unity Catalog
- Want to minimize operational overhead
- Cost-sensitive workloads with variable load

**Use Kafka when:**
- Multiple consumers need the same event stream
- High-throughput pub/sub architecture
- Need message replay from arbitrary offsets
- Complex event routing and filtering